In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []
for file in files:
    doc = file.parse()
    documents.append(doc)

print(f"Total lesson pages: {len(documents)}")
print(f"\nFirst document keys: {documents[0].keys()}")
print(f"First document filename: {documents[0]['filename']}")

Total lesson pages: 72

First document keys: dict_keys(['content', 'filename'])
First document filename: 01-agentic-rag/lessons/01-intro.md


In [2]:
import minsearch

index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)
print("Index ready ")

Index ready 


In [3]:
results = index.search(
    query="How does the agentic loop keep calling the model until it stops?",
    num_results=5
)

print(f"First result filename: {results[0]['filename']}")
print(f"\nAll results:")
for r in results:
    print(f"- {r['filename']}")

First result filename: 01-agentic-rag/lessons/14-agentic-loop.md

All results:
- 01-agentic-rag/lessons/14-agentic-loop.md
- 01-agentic-rag/lessons/15-frameworks.md
- 01-agentic-rag/lessons/13-function-calling.md
- 01-agentic-rag/lessons/11-agents-intro.md
- 01-agentic-rag/lessons/16-other-frameworks.md


In [8]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

client = OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

def search(query, num_results=5):
    return index.search(query=query, num_results=num_results)

def build_context(search_results):
    context = ""
    for doc in search_results:
        context += f"filename: {doc['filename']}\ncontent: {doc['content']}\n\n"
    return context.strip()

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = f"""
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the course lessons.
Use only the facts from the CONTEXT when answering.
If the answer is not in the context, say "I don't know."

QUESTION: {question}

CONTEXT:
{context}
""".strip()
    return prompt

def rag(question):
    search_results = search(question, num_results=2) 
    prompt = build_prompt(question, search_results)
    
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    
    answer = response.choices[0].message.content
    input_tokens = response.usage.prompt_tokens
    
    return answer, input_tokens

query = "How does the agentic loop keep calling the model until it stops?"
answer, input_tokens = rag(query)

print(f"Answer: {answer}\n")
print(f"Input tokens: {input_tokens}")

Answer: I do not know how the agentic loop keeps calling the model until it stops based on the provided context since the agentic loop's stopping condition is not mentioned, only that it stops when a response has no function calls.

Input tokens: 3711


In [9]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

print(f"Total chunks: {len(chunks)}")
print(f"\nFirst chunk keys: {chunks[0].keys()}")
print(f"First chunk filename: {chunks[0]['filename']}")
print(f"First chunk start: {chunks[0]['start']}")
print(f"First chunk content length: {len(chunks[0]['content'])}")

Total chunks: 295

First chunk keys: dict_keys(['start', 'content', 'filename'])
First chunk filename: 01-agentic-rag/lessons/01-intro.md
First chunk start: 0
First chunk content length: 2000


In [11]:
# index chunks instead of full documents
chunk_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
chunk_index.fit(chunks)
print("Chunk index ready ")

def search_chunks(query, num_results=5):
    return chunk_index.search(query=query, num_results=num_results)

def rag_chunked(question):
    search_results = search_chunks(question, num_results=2)
    prompt = build_prompt(question, search_results)
    
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    
    answer = response.choices[0].message.content
    input_tokens = response.usage.prompt_tokens
    
    return answer, input_tokens

query = "How does the agentic loop keep calling the model until it stops?"
answer, input_tokens = rag_chunked(query)

print(f"Answer: {answer}\n")
print(f"Input tokens (chunked): {input_tokens}")

Chunk index ready 
Answer: The agentic loop keeps calling the model until it stops. This happens because the while loop has the condition `while True`, which continuously loops until a `break` statement is encountered. In this case, the `break` statement is encountered when `has_function_calls == False`, indicating that the model has returned a response without any function calls.

Input tokens (chunked): 1031


In [24]:
import json

search_call_count = 0

tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "search",
            "description": "Search the course lessons knowledge base for relevant content.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query string"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

def search(query: str) -> list:
    global search_call_count
    search_call_count += 1
    print(f"Search #{search_call_count}: {query}")
    results = chunk_index.search(query=query, num_results=3)
    return [{'filename': r['filename'], 'content': r['content'][:500]} for r in results]

def run_agent(question):
    messages = [
        {"role": "system", "content": "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."},
        {"role": "user", "content": question}
    ]
    
    while True:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=messages,
            tools=tools_schema
        )
        
        message = response.choices[0].message
        finish_reason = response.choices[0].finish_reason
        messages.append(message)
        
        if finish_reason == "stop":
            return message.content
        
        if finish_reason == "tool_calls":
            for tool_call in message.tool_calls:
                args = json.loads(tool_call.function.arguments)
                result = search(**args)
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(result)
                })

question = "How does the agentic loop work, and how is it different from plain RAG?"
answer = run_agent(question)

print(f"\nFinal answer: {answer}")
print(f"\nTotal search calls: {search_call_count}")

Search #1: agentic loop vs RAG
Search #2: agentic loop RAG differences
Search #3: agentic loop vs plan RAG

Final answer: The agentic loop is a design pattern that represents the core of any AI agent. It consists of a while loop that calls the large language model (LLM), executes any tool calls it returns, sends the results back, and stops when the model produces a final answer with no more tool calls.

In contrast, a plain RAG (Retrieval-Augmented Generator) is a pipeline with three steps: search, build prompt, and LLM. The search step uses keyword search, but it can be swapped with vector search.

The main difference between the agentic loop and a plain RAG is that the agentic loop is a more general and modular design pattern, which can handle the loop for you, whereas a plain RAG is a specific implementation of the loop with a specific search step.

Therefore, the agentic loop is more flexible and can be used as a foundation for various agent frameworks, while a plain RAG is a speci